# SkyEye — 天气分类
## EfficientNet-B5 → 知识蒸馏 → B0 → 结构化剪枝 → ONNX 导出

按顺序执行以下 Cell：

| Cell | 内容 |
|------|------|
| 1 | 安装依赖 |
| 2 | 数据准备（探索 + 复制 + 解压） |
| 3 | 数据合并 + DataLoader |
| 4 | 训练教师模型 (EfficientNet-B5) |
| 5 | 知识蒸馏 (B5 → B0) |
| 6 | 结构化剪枝 + 微调 |
| 7 | ONNX 导出 + INT8 量化 + 测速 |
| 8 | (可选) CPU 单张推理测试 |

In [ ]:
# Cell 1: 安装依赖（云端 PyTorch / torchvision 已预装，跳过）
!pip install timm==1.0.8 onnx==1.16.1 onnxruntime-gpu==1.18.1 tqdm numpy==1.26.4 scipy==1.12.0 scikit-learn==1.5.2

In [ ]:
# ============================================================
# Cell 2: 数据准备 — 探索 + 复制 + 解压
# 自动发现 datasets/ 中的数据源，复制/解压到 _data/ 可写目录
# ============================================================
import os

# Mo 平台数据集目录
datasets_root = "datasets"
if not os.path.isdir(datasets_root):
    datasets_root = "/home/jovyan/work/datasets"

print("=" * 60)
print("数据准备：探索 → 复制 → 解压")
print("=" * 60)

# 确保 _data 父目录存在
os.makedirs("_data", exist_ok=True)

# --- Step 1: 探索数据源 ---
weather_src = None
zip_src = None

for import_name in sorted(os.listdir(datasets_root)):
    import_dir = os.path.join(datasets_root, import_name)
    if not os.path.isdir(import_dir) or import_name.startswith('.'):
        continue
    for item in sorted(os.listdir(import_dir)):
        item_path = os.path.join(import_dir, item)
        if os.path.isdir(item_path) and item == "weather_classification":
            weather_src = item_path
            for cls in sorted(os.listdir(item_path)):
                cls_path = os.path.join(item_path, cls)
                if os.path.isdir(cls_path):
                    count = len([f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))])
                    print(f"  {item}/{cls}/ ({count} 张)")
        elif item.lower().endswith('.zip'):
            zip_src = item_path
            size_mb = os.path.getsize(item_path) / (1024 * 1024)
            print(f"  {item} ({size_mb:.1f} MB)")

# --- Step 2: 复制 weather_classification ---
if weather_src and not os.path.exists("_data/weather_src1"):
    print(f"\n复制: {weather_src} → _data/weather_src1 ...")
    !cp -R {weather_src} _data/weather_src1
elif os.path.exists("_data/weather_src1"):
    print("\n_data/weather_src1 已存在，跳过复制")
else:
    print("\n⚠ 未找到 weather_classification")

# --- Step 3: 解压 zip（含外壳目录处理）---
if zip_src and not os.path.exists("_data/weather_src2"):
    print(f"\n解压: {zip_src} → _data/weather_src2 ...")
    os.makedirs("_data/weather_src2", exist_ok=True)
    !7zx {zip_src} -o_data/weather_src2
    # 处理外壳目录：zip 内可能有一层 wrapper，自动进入
    members = [f for f in os.listdir("_data/weather_src2")
               if os.path.isdir(os.path.join("_data/weather_src2", f)) and not f.startswith('_')]
    if len(members) == 1 and not any(
            os.path.isfile(os.path.join("_data/weather_src2", f)) for f in os.listdir("_data/weather_src2")):
        # 单层外壳 → 重命名
        inner = os.path.join("_data/weather_src2", members[0])
        tmp = "_data/weather_src2_tmp"
        os.rename(inner, tmp)
        os.rmdir("_data/weather_src2")
        os.rename(tmp, "_data/weather_src2")
        print(f"  检测到外壳目录: {members[0]}，已去除")
    print(f"  内容: {sorted(os.listdir('_data/weather_src2'))}")
elif os.path.exists("_data/weather_src2"):
    print("\n_data/weather_src2 已存在，跳过解压")
else:
    print("\n⚠ 未找到 data_split.zip")

print(f"\n{'='*60}")
print("✓ 数据准备完成")
print(f"  weather_src: {weather_src}")
print(f"  zip_src: {zip_src}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 3: 数据合并 + 创建 DataLoader
# 多源合并到 _data/weather，分层划分训练/验证集
# ============================================================
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split

from config import CONFIG

# --- Step 1: 数据合并 ---
data_sources = []
if os.path.exists("_data/weather_src1"):
    data_sources.append({
        "path": "_data/weather_src1",
        "class_map": {"haze": "foggy", "snow": "snowy", "thunder": "thundery"}
    })
if os.path.exists("_data/weather_src2"):
    data_sources.append({
        "path": "_data/weather_src2",
        "class_map": {}
    })

if not data_sources:
    raise RuntimeError("❌ 没有可用的数据源！请先运行 Cell 2 并确认数据集已导入")

CONFIG["data_roots"] = data_sources
print(f"数据源: {[s['path'] for s in data_sources]}")

from data.dataset import prepare_data
data_root = prepare_data()
print(f"数据已就绪: {data_root}")

# --- Step 2: 环境信息 ---
print(f"\nDevice: {CONFIG['device']}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Classes: {CONFIG['class_names']}")
print(f"Teacher: {CONFIG['teacher_model']}, Student: {CONFIG['student_model']}")

# --- Step 3: 创建 DataLoader ---
full_dataset = ImageFolder(data_root)
num_classes = len(full_dataset.classes)
class_counts = np.bincount(full_dataset.targets, minlength=num_classes)

indices = np.arange(len(full_dataset))
train_idx, val_idx = train_test_split(
    indices, test_size=CONFIG["val_split"],
    stratify=full_dataset.targets, random_state=CONFIG["seed"],
)

from data.augmentations import get_train_transforms, get_val_transforms

train_ds = torch.utils.data.Subset(
    ImageFolder(data_root, transform=get_train_transforms(CONFIG["img_size"])), train_idx)
val_ds = torch.utils.data.Subset(
    ImageFolder(data_root, transform=get_val_transforms(CONFIG["img_size"])), val_idx)

nw = CONFIG["num_workers"]
pin = torch.cuda.is_available()

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=nw, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                        num_workers=nw, pin_memory=pin)

print(f"\nClass distribution: {dict(zip(full_dataset.classes, class_counts.astype(int)))}")
print(f"Train: {len(train_ds)} samples, Val: {len(val_ds)} samples")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# 类别权重（供 FocalLoss 使用）
from data.dataset import compute_class_weights
class_weights = compute_class_weights(class_counts)

print(f"\n{'='*60}")
print("✓ 数据合并 + DataLoader 创建完成")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 4: 训练教师模型 (EfficientNet-B5)
# FocalLoss + 混合精度 + Cosine 调度
# 输出：results/teacher_best.pth
# ============================================================
from training.train_teacher import train_teacher
teacher = train_teacher()
print("\n" + "=" * 60)
print("✓ 教师模型训练完成 -> results/teacher_best.pth")
print("=" * 60)

In [ ]:
# ============================================================
# Cell 5: 知识蒸馏 (B5 → B0)
# 软标签蒸馏 + 中间层特征对齐
# 输出：results/student_distilled_best.pth
# ============================================================
from training.distill_student import run_distillation
student = run_distillation()
print("\n" + "=" * 60)
print("✓ 知识蒸馏完成 -> results/student_distilled_best.pth")
print("=" * 60)

In [ ]:
# ============================================================
# Cell 6: 结构化剪枝 + 渐进微调
# 2 轮渐进剪枝 (20% → 40%)，每轮后微调，最终固化 mask
# 输出：results/student_pruned_final.pth
# ============================================================
from training.prune_finetune import prune_and_finetune
pruned_model = prune_and_finetune()
print("\n" + "=" * 60)
print("✓ 剪枝微调完成 -> results/student_pruned_final.pth")
print("=" * 60)

In [ ]:
# ============================================================
# Cell 7: ONNX 导出 + INT8 量化 + CPU 推理测速
# PyTorch → ONNX FP32 → INT8 动态量化 → CPU Benchmark
# 输出：results/weather_model.onnx, results/weather_model_int8.onnx
# ============================================================
from inference.export_onnx import export_to_onnx, quantize_to_int8, benchmark_cpu
onnx_path = export_to_onnx('results/student_pruned_final.pth')
int8_path = quantize_to_int8(onnx_path)
benchmark_cpu(onnx_path, int8_path)
print("\n" + "=" * 60)
print("✓ ONNX 导出 + 量化 + 测速完成")
print("=" * 60)

In [ ]:
# ============================================================
# Cell 8 (可选): CPU 单张推理测试
# 使用剪枝后的模型进行推理验证
# ============================================================
from inference.infer import predict_image
result = predict_image('test_image.jpg')  # 替换为实际图片路径
print(f'Prediction: {result["prediction"]} (Confidence: {result["confidence"]:.4f})')
for name, prob in result['top_k']:
    print(f'  {name}: {prob:.4f}')
print("\n" + "=" * 60)
print("✓ 推理测试完成")
print("=" * 60)